## Model Benchmark Tracker
### Project 2 Data Portfolio

This notebook demonstrates a complete ETL pipeline using real public data from the 
Open LLM Leaderboard which is a community benchmark repository tracking AI model performance 
across standardized evaluation tasks.

**Extract:** Pull structured model evaluation data from a public dataset source  
**Transform:** Normalize scores, flag missing values, reshape into analysis-ready format  
**Load:** Write clean output to a local SQLite database and CSV export

**Skills demonstrated:** ETL pipeline design, API/dataset ingestion, data normalization, 
database writes, exploratory data analysis

In [13]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "pandas", "numpy"])

0

In [14]:
import pandas as pd
import numpy as np
import sqlite3

# Pull real benchmark data from HuggingFace public dataset
from datasets import load_dataset

dataset = load_dataset("open-llm-leaderboard/contents", split="train")
df_raw = dataset.to_pandas()

print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumns: {list(df_raw.columns)}")
df_raw.head(5)

Dataset shape: (4576, 36)

Columns: ['eval_name', 'Precision', 'Type', 'T', 'Weight type', 'Architecture', 'Model', 'fullname', 'Model sha', 'Average ⬆️', 'Hub License', 'Hub ❤️', '#Params (B)', 'Available on the hub', 'MoE', 'Flagged', 'Chat Template', 'CO₂ cost (kg)', 'IFEval Raw', 'IFEval', 'BBH Raw', 'BBH', 'MATH Lvl 5 Raw', 'MATH Lvl 5', 'GPQA Raw', 'GPQA', 'MUSR Raw', 'MUSR', 'MMLU-PRO Raw', 'MMLU-PRO', 'Merged', 'Official Providers', 'Upload To Hub Date', 'Submission Date', 'Generation', 'Base Model']


,eval_name,Precision,Type,T,Weight type,Architecture,Model,fullname,Model sha,Average ⬆️,...,MUSR Raw,MUSR,MMLU-PRO Raw,MMLU-PRO,Merged,Official Providers,Upload To Hub Date,Submission Date,Generation,Base Model
0,0-hero_Matter-0.2-7B-DPO_bfloat16,bfloat16,"💬 chat models (RLHF, DPO, IFT, ...)",💬,Original,MistralForCausalLM,"<a target=""_blank"" href=""https://huggingface.c...",0-hero/Matter-0.2-7B-DPO,26a66f0d862e2024ce4ad0a09c37052ac36e8af6,8.906361,...,0.381375,5.871875,0.116356,1.817376,False,False,2024-04-13,2024-08-05,0,0-hero/Matter-0.2-7B-DPO
1,01-ai_Yi-1.5-34B_bfloat16,bfloat16,🟢 pretrained,🟢,Original,LlamaForCausalLM,"<a target=""_blank"" href=""https://huggingface.c...",01-ai/Yi-1.5-34B,4b486f81c935a2dadde84c6baa1e1370d40a098f,25.646494,...,0.423604,11.217188,0.466589,40.732122,False,True,2024-05-11,2024-06-12,0,01-ai/Yi-1.5-34B
2,01-ai_Yi-1.5-34B-32K_bfloat16,bfloat16,🟢 pretrained,🟢,Original,LlamaForCausalLM,"<a target=""_blank"" href=""https://huggingface.c...",01-ai/Yi-1.5-34B-32K,2c03a29761e4174f20347a60fbe229be4383d48b,26.727913,...,0.439823,14.077865,0.470911,41.212323,False,True,2024-05-15,2024-06-12,0,01-ai/Yi-1.5-34B-32K
3,01-ai_Yi-1.5-34B-Chat_bfloat16,bfloat16,"💬 chat models (RLHF, DPO, IFT, ...)",💬,Original,LlamaForCausalLM,"<a target=""_blank"" href=""https://huggingface.c...",01-ai/Yi-1.5-34B-Chat,f3128b2d02d82989daae566c0a7eadc621ca3254,33.357994,...,0.428198,13.058073,0.452045,39.116061,False,True,2024-05-10,2024-06-12,0,01-ai/Yi-1.5-34B-Chat
4,01-ai_Yi-1.5-34B-Chat-16K_bfloat16,bfloat16,"💬 chat models (RLHF, DPO, IFT, ...)",💬,Original,LlamaForCausalLM,"<a target=""_blank"" href=""https://huggingface.c...",01-ai/Yi-1.5-34B-Chat-16K,ff74452e11f0f749ab872dc19b1dd3813c25c4d8,29.403555,...,0.439760,13.736719,0.454455,39.383865,False,True,2024-05-15,2024-07-15,0,01-ai/Yi-1.5-34B-Chat-16K


## Step 1: Inspect the Raw Data

Real-world benchmark data arrives with inconsistent types, missing values, 
and columns that need reshaping for analysis.

**Key columns:**
- `Model` — model name/identifier
- `Average ⬆️` — overall benchmark average score
- `IFEval`, `BBH`, `MATH Lvl 5`, `GPQA`, `MUSR`, `MMLU-PRO` — individual benchmark scores
- `#Params (B)` — model size in billions of parameters
- `Flagged` — whether the submission was flagged for review

**Issues to address:**
- Mixed types in score columns
- Missing values across benchmark scores
- Irrelevant columns that add noise
- Dates stored as strings

In [15]:
# Select relevant columns only
cols = ['Model', 'Average ⬆️', 'IFEval', 'BBH', 'MATH Lvl 5', 
        'GPQA', 'MUSR', 'MMLU-PRO', '#Params (B)', 
        'Flagged', 'Submission Date', 'Architecture']

df = df_raw[cols].copy()

print("Null counts per column:")
print(df.isnull().sum())

Null counts per column:
Model              0
Average ⬆️         0
IFEval             0
BBH                0
MATH Lvl 5         0
GPQA               0
MUSR               0
MMLU-PRO           0
#Params (B)        0
Flagged            0
Submission Date    0
Architecture       0
dtype: int64


In [16]:
# Check data types and look for hidden issues
print(df.dtypes)
print(f"\nScore range for Average:")
print(df['Average ⬆️'].describe())
print(f"\nFlagged value counts:")
print(df['Flagged'].value_counts())
print(f"\nUnique architectures: {df['Architecture'].nunique()}")
print(f"\nSample submission dates:")
print(df['Submission Date'].head(5))

Model                  str
Average ⬆️         float64
IFEval             float64
BBH                float64
MATH Lvl 5         float64
GPQA               float64
MUSR               float64
MMLU-PRO           float64
#Params (B)        float64
Flagged               bool
Submission Date        str
Architecture           str
dtype: object

Score range for Average:
count    4576.000000
mean       21.814863
std        10.802092
min         0.737851
25%        13.936474
50%        21.950426
75%        29.258424
max        52.081384
Name: Average ⬆️, dtype: float64

Flagged value counts:
Flagged
False    4575
True        1
Name: count, dtype: int64

Unique architectures: 64

Sample submission dates:
0    2024-08-05
1    2024-06-12
2    2024-06-12
3    2024-06-12
4    2024-07-15
Name: Submission Date, dtype: str


## Step 2: Transform the Data

Key transformations:
- Convert Submission Date to datetime
- Flag suspiciously low scores (below 5.0) as outliers
- Isolate flagged model submissions for review
- Rename columns for cleaner output

In [17]:
# Convert date
df['Submission Date'] = pd.to_datetime(df['Submission Date'])

# Rename columns for cleaner output
df = df.rename(columns={
    'Average ⬆️': 'avg_score',
    '#Params (B)': 'params_billions',
    'Submission Date': 'submission_date',
    'Architecture': 'architecture',
    'Flagged': 'flagged',
    'Model': 'model'
})

# Flag low scoring outliers
df['low_score_flag'] = df['avg_score'] < 5.0

print(f"Low scoring models flagged: {df['low_score_flag'].sum()}")
print(f"Flagged submissions: {df['flagged'].sum()}")
print(f"\nSample cleaned data:")
df.head(5)

Low scoring models flagged: 280
Flagged submissions: 1

Sample cleaned data:


,model,avg_score,IFEval,BBH,MATH Lvl 5,GPQA,MUSR,MMLU-PRO,params_billions,flagged,submission_date,architecture,low_score_flag
0,"<a target=""_blank"" href=""https://huggingface.c...",8.906361,33.027921,10.055525,1.435045,1.230425,5.871875,1.817376,7.242,False,2024-08-05,MistralForCausalLM,False
1,"<a target=""_blank"" href=""https://huggingface.c...",25.646494,28.411725,42.749363,15.332326,15.436242,11.217188,40.732122,34.389,False,2024-06-12,LlamaForCausalLM,False
2,"<a target=""_blank"" href=""https://huggingface.c...",26.727913,31.186917,43.381847,15.407855,15.100671,14.077865,41.212323,34.389,False,2024-06-12,LlamaForCausalLM,False
3,"<a target=""_blank"" href=""https://huggingface.c...",33.357994,60.667584,44.262826,27.719033,15.324385,13.058073,39.116061,34.389,False,2024-06-12,LlamaForCausalLM,False
4,"<a target=""_blank"" href=""https://huggingface.c...",29.403555,45.645000,44.536157,21.374622,11.744966,13.736719,39.383865,34.389,False,2024-07-15,LlamaForCausalLM,False


In [18]:
import re

# Extract clean model name from HTML link
def clean_model_name(val):
    match = re.search(r'>([^<]+)</a>', str(val))
    if match:
        return match.group(1).strip()
    return val.strip()

df['model'] = df['model'].apply(clean_model_name)

print("Sample model names after cleaning:")
print(df['model'].head(10).tolist())

Sample model names after cleaning:
['0-hero/Matter-0.2-7B-DPO', '01-ai/Yi-1.5-34B', '01-ai/Yi-1.5-34B-32K', '01-ai/Yi-1.5-34B-Chat', '01-ai/Yi-1.5-34B-Chat-16K', '01-ai/Yi-1.5-6B', '01-ai/Yi-1.5-6B-Chat', '01-ai/Yi-1.5-9B', '01-ai/Yi-1.5-9B-32K', '01-ai/Yi-1.5-9B-Chat']


In [19]:
# --- Load: Write to SQLite database ---
conn = sqlite3.connect("benchmark_tracker.db")

df.to_sql("model_benchmarks", conn, if_exists="replace", index=False)

print("Data written to SQLite database: benchmark_tracker.db")
print(f"Table: model_benchmarks")
print(f"Rows written: {len(df)}")

# Verify with a quick query
query_result = pd.read_sql("SELECT model, avg_score, architecture FROM model_benchmarks ORDER BY avg_score DESC LIMIT 10", conn)
print("\nTop 10 models by average score:")
query_result

Data written to SQLite database: benchmark_tracker.db
Table: model_benchmarks
Rows written: 4576

Top 10 models by average score:


,model,avg_score,architecture
0,MaziyarPanahi/calme-3.2-instruct-78b,52.081384,Qwen2ForCausalLM
1,MaziyarPanahi/calme-3.1-instruct-78b,51.287490,Qwen2ForCausalLM
2,dfurman/CalmeRys-78B-Orpo-v0.1,51.231323,Qwen2ForCausalLM
3,MaziyarPanahi/calme-2.4-rys-78b,50.765047,Qwen2ForCausalLM
4,huihui-ai/Qwen2.5-72B-Instruct-abliterated,48.106471,Qwen2ForCausalLM
5,Qwen/Qwen2.5-72B-Instruct,47.980460,Qwen2ForCausalLM
6,MaziyarPanahi/calme-2.1-qwen2.5-72b,47.856722,Qwen2ForCausalLM
7,newsbang/Homer-v1.0-Qwen2.5-72B,47.464376,Qwen2ForCausalLM
8,ehristoforu/qwen2.5-test-32b-it,47.368357,Qwen2ForCausalLM
9,Saxo/Linkbricks-Horizon-AI-Avengers-V1-32B,47.336956,Qwen2ForCausalLM


In [20]:
# Export clean dataset to CSV
df.to_csv("benchmark_tracker_clean.csv", index=False)
print("Clean dataset exported to benchmark_tracker_clean.csv")

# Export flagged models separately
df_flagged = df[df['flagged'] == True]
df_low = df[df['low_score_flag'] == True]

print(f"\nFlagged submissions: {len(df_flagged)}")
print(f"Low score outliers (avg < 5.0): {len(df_low)}")
print("\nFlagged model:")
df_flagged[['model', 'avg_score', 'submission_date']]

Clean dataset exported to benchmark_tracker_clean.csv

Flagged submissions: 1
Low score outliers (avg < 5.0): 280

Flagged model:


,model,avg_score,submission_date
3655,newsbang/Homer-v0.5-Qwen2.5-7B,34.763845,2024-11-20


## Summary of Findings

| Step | Action | Result |
|---|---|---|
| Extract | Loaded Open LLM Leaderboard dataset | 4,576 models, 36 columns |
| Transform | Cleaned HTML from model names | Readable model identifiers |
| Transform | Converted submission dates | Standardized to datetime |
| Transform | Flagged low scoring outliers (avg < 5.0) | 280 models flagged |
| Transform | Isolated flagged submissions | 1 model flagged for review |
| Load | Written to SQLite database | 4,576 rows in model_benchmarks table |
| Load | Exported to CSV | benchmark_tracker_clean.csv |

**Key finding:** 6.1% of submitted models scored below 5.0 average — 
suggesting either incomplete evaluation runs or low-quality submissions 
that warrant closer review before inclusion in downstream analysis.

**Top architecture:** Qwen2ForCausalLM dominates the top 10 by average score.